# STAT 764 · Meeting 4 — The leakage laboratory

**Thursday, September 17 · Concept 10 min · Studio 45 min**

Everything so far has been about producing an honest estimate. Today is about
the ways an estimate stops being honest without anyone doing anything wrong on
purpose.

This is the meeting the rest of the semester refers back to.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (GroupKFold, KFold, StratifiedKFold,
                                     TimeSeriesSplit, cross_val_score)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline

bike = load("bike_hour.csv")
print(f"Capital Bikeshare, hourly: {bike.shape[0]:,} hours, {bike.shape[1]} columns")
bike.head(3)

## 1. A perfect model

The task: predict `cnt`, the number of bikes rented in an hour. We have twelve
candidate predictors and no reason to prefer any of them, so let's start by
throwing in everything and seeing where we stand.

This is a completely ordinary thing to do, and people do it every day.

In [ ]:
X_everything = bike.drop(columns=["cnt", "dteday"])
y = bike["cnt"]
cv = KFold(5, shuffle=True, random_state=764)

score = cross_val_score(LinearRegression(), X_everything, y, cv=cv, scoring="r2").mean()
print(f"  5-fold CV R-squared: {score:.6f}")

**1.000000.** Cross-validated, on held-out folds, with a pipeline. Every
discipline from Meeting 3 applied correctly.

Before reading on: what would you do if you got this?

The right instinct is not delight. It is *suspicion* — nothing in the world is
predicted perfectly, so a perfect score is evidence about your data, not about
your model.

In [ ]:
print((bike["casual"] + bike["registered"] == bike["cnt"]).all())
print(f"\ncasual + registered = cnt, in all {len(bike):,} rows.")
print("The outcome was among the predictors. We asked the model to predict a")
print("total from its own two parts.")

In [ ]:
X_honest = X_everything.drop(columns=["casual", "registered"])
honest = cross_val_score(LinearRegression(), X_honest, y, cv=cv, scoring="r2").mean()
print(f"  with the two components removed: R-squared {honest:.3f}")

From 1.000 to 0.388. The first number was not *wrong* — the model really does
predict `cnt` that well from `casual` and `registered`. It is **useless**,
because at the moment you would need a forecast, you do not know either of
them. They are not available until the hour is over.

> **Leakage:** information is available to the model during training that will
> not be available at the moment the prediction actually has to be made.

Note that no rule of resampling was broken here. Cross-validation cannot
protect you from this, because the problem is in what the columns *mean*, and
cross-validation does not know what anything means.

## 2. Leakage you cannot see by reading column names

The bike case is the easy one — someone could have caught it by thinking about
the variables for a minute. Now a harder one, where nothing in the data is
suspicious at all.

Below, `y` is generated **completely independently of `X`**. There is no signal.
The true predictive R-squared is exactly zero, and we know this because we
wrote the data-generating process ourselves.

We do one ordinary thing first: with 5,000 candidate predictors and only 200
observations, we screen down to the 20 most correlated with the outcome. Then
we cross-validate properly.

In [ ]:
rng = np.random.default_rng(764)
n, p = 200, 5000
X_noise = rng.normal(size=(n, p))
y_noise = rng.normal(size=n)            # independent of X_noise, by construction

correlations = np.abs(np.array([np.corrcoef(X_noise[:, j], y_noise)[0, 1] for j in range(p)]))
top20 = np.argsort(correlations)[-20:]

leaky = cross_val_score(Ridge(), X_noise[:, top20], y_noise,
                        cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
print(f"  screen on all data, then cross-validate:  R-squared {leaky:.3f}")

**0.435, from data containing no signal whatsoever.**

The screening step looked at the outcomes of all 200 observations — including
the ones that would later serve as test folds — and kept the 20 columns that
happened to correlate with them by chance. Those coincidences were then still
present in the test folds, because the test folds are where the coincidences
were found.

The cross-validation was real. The pipeline was real. The result is an
artifact of a decision made one line earlier.

The fix is the same structural one as Meeting 2: put the screening **inside**
the pipeline, so it is refit on each training fold and never sees the fold it
will be scored on.

In [ ]:
honest_pipe = Pipeline([
    ("screen", SelectKBest(f_regression, k=20)),
    ("model", Ridge()),
])
clean = cross_val_score(honest_pipe, X_noise, y_noise,
                        cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
print(f"  screening inside each fold:               R-squared {clean:.3f}")
print("\n  Negative, i.e. worse than predicting the mean — which is the correct")
print("  answer for data with no signal in it.")

## 3. The resampling myth

The outcome is rare — 5% of hours. Someone will tell you to balance the classes
before modeling. Very often they will tell you to do it to the whole dataset.

In [ ]:
peak = (bike["cnt"] > bike["cnt"].quantile(0.95)).astype(int)
X_peak = bike[["hr", "temp", "atemp", "hum", "windspeed", "workingday",
               "weathersit", "season", "yr", "mnth", "holiday", "weekday"]]
print(f"  prevalence of a 'peak demand' hour: {peak.mean():.3f}")


def oversample(X_part, y_part, seed=0):
    """Duplicate minority rows at random until the classes are balanced."""
    r = np.random.default_rng(seed)
    minority = np.where(y_part == 1)[0]
    majority = np.where(y_part == 0)[0]
    extra = r.choice(minority, size=len(majority) - len(minority), replace=True)
    keep = np.concatenate([majority, minority, extra])
    return X_part.iloc[keep], y_part.iloc[keep]

In [ ]:
# The common recipe: balance the dataset, then cross-validate it.
X_bal, y_bal = oversample(X_peak, peak)
myth = cross_val_score(KNeighborsClassifier(1), X_bal, y_bal,
                       cv=StratifiedKFold(5, shuffle=True, random_state=0),
                       scoring="roc_auc").mean()
print(f"  balance first, then cross-validate:  AUC {myth:.3f}")

In [ ]:
# The honest version: balance inside each training fold only.
aucs = []
for train_idx, test_idx in StratifiedKFold(5, shuffle=True, random_state=0).split(X_peak, peak):
    X_tr, y_tr = oversample(X_peak.iloc[train_idx], peak.iloc[train_idx])
    fitted = KNeighborsClassifier(1).fit(X_tr, y_tr)
    aucs.append(roc_auc_score(peak.iloc[test_idx],
                              fitted.predict_proba(X_peak.iloc[test_idx])[:, 1]))
print(f"  balance inside each training fold:   AUC {np.mean(aucs):.3f}")
print(f"\n  The first number is inflated by {myth - np.mean(aucs):.3f}.")

The mechanism is worth saying out loud, because it is completely mechanical.

Oversampling makes **copies of rows**. Cross-validation then splits those rows
at random — so a row and its own duplicate routinely land in different folds. A
1-nearest-neighbor classifier asked about a test row finds an exact copy of it
sitting in the training fold and answers perfectly.

It is not learning about peak demand. It is learning that it has seen this
exact row before.

The same thing happens with SMOTE, which manufactures synthetic minority points
by interpolating between real ones — the synthetic points are not independent
of the real ones they were built from. **Any resampling of the data must happen
inside the training fold, never before the split.**

---

## Studio — triage

### Part A · as a team, 20 minutes

For each workflow: **LEAK / NO LEAK / DEPENDS**, and if it leaks, say what
information reaches the model that would not exist at prediction time, and
whether you would expect the effect to be large or small.

| # | workflow |
|---|---|
| 1 | Standardize every feature using the mean and SD of the full dataset, then split into train and test. |
| 2 | Log-transform the outcome before splitting, because it is right-skewed. |
| 3 | Fill missing values with the column median computed over the full dataset, then cross-validate. |
| 4 | With 400 candidate predictors, keep the 20 most correlated with the outcome, then cross-validate. |
| 5 | The positive class is 3% of rows, so oversample it to 50% before cross-validating. |
| 6 | Try 200 hyperparameter settings by cross-validation, then report the best CV score as your estimate of performance. |
| 7 | Hospital data with several encounters per patient. Split rows at random into train and test. |
| 8 | Predict next quarter's revenue. Split the historical quarters at random into train and test. |

Write your verdicts in the cell below **before** you run anything.

### Part B · 25 minutes

Pick the **two your team argued about most** and stop arguing — measure them.
You have the pattern three times above: build the leaky version, build the
honest version, report the gap.

The size of a leak is a number. Treat it as one.

§1–§3 are your templates: workflows **1, 3, 4 and 5** are all the same move —
take the suspect step inside the fold. **#6, #7 and #8** are a different animal,
and their fixes are tools we build later, so for those the answer is your verdict
plus the *name* of the fix. Runnable starters for all three are in the **Stretch**
cells at the very end, if a workflow you argued about is one of them.

### Team verdicts

*(double-click to edit)*

| # | verdict | what leaks, and how big |
|---|---|---|
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |
| 7 | | |
| 8 | | |

In [ ]:
# YOUR CODE HERE — measure the two you disagreed about.

### Stretch · the three that need tools we have not built yet

#6, #7 and #8 are not in the "move the step inside the fold" family, so §1–§3 are
not templates for them — you meet the real fixes later (nested cross-validation on
Oct 1; grouped and time-ordered splits after that). If your team wants to *see*
these rather than only name them, here are runnable starters.

**A note on the data.** Ames and the bike file have no repeated patient and no
usable time axis — so there is nothing in them to leak through a group or a date.
To watch those two leaks we **build data that has the structure on purpose**: a
handful of patients with several rows each, or a series that runs in time. Then we
split it the wrong way and the right way and read off the gap. Manufacturing the
smallest dataset that makes an effect visible is itself a skill — it is how you
test a claim about leakage when no real dataset is in front of you.

**#6 · the winner's curse.** Nothing crosses the train/test line — every score
below is an honest cross-validation. The bias is entirely in *reporting the
maximum* of many tries; the luckiest of 200 noisy estimates is not an unbiased
estimate of anything. (The honest fix, nested CV, is Oct 1.)

In [ ]:
rng = np.random.default_rng(6)
Xw = rng.normal(size=(200, 100))              # 100 candidate predictors
yw = rng.normal(size=200)                     # unrelated to every one of them: the truth is 0
kfw = KFold(5, shuffle=True, random_state=1)

scores = [cross_val_score(LinearRegression(), Xw[:, rng.choice(100, 5, replace=False)],
                          yw, cv=kfw, scoring="r2").mean() for _ in range(200)]
print(f"  best of 200 honest CV scores: {max(scores):+.3f}   (the truth is 0)")
print(f"  mean of the 200:              {np.mean(scores):+.3f}")

**#7 · repeated patients.** We *create* 200 patients, give each a constant
"fingerprint" (features that repeat across their encounters) and an outcome that
depends only on who they are. A random split drops other encounters of the same
patient into the training rows and the model recognizes the patient; `GroupKFold`
holds whole patients out, and the fingerprint becomes useless.

In [ ]:
rng = np.random.default_rng(7)
enc = rng.integers(2, 6, size=200)                        # how many encounters each patient has
pid = np.repeat(np.arange(200), enc)                      # the patient id attached to every row
fingerprint = (np.repeat(rng.normal(size=(200, 5)), enc, axis=0)
               + rng.normal(scale=0.01, size=(len(pid), 5)))
y_pat = np.repeat(rng.normal(size=200), enc) + rng.normal(scale=0.1, size=len(pid))

random_split = cross_val_score(KNeighborsRegressor(5), fingerprint, y_pat,
                               cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
grouped = cross_val_score(KNeighborsRegressor(5), fingerprint, y_pat, groups=pid,
                          cv=GroupKFold(5), scoring="r2").mean()
print(f"  random row split: R2 {random_split:+.3f}     GroupKFold on patient: R2 {grouped:+.3f}")

**#8 · predicting the future.** We *create* a series that runs in time — a random
walk. A random split lets the model interpolate between past and future points; a
time-ordered split (`TimeSeriesSplit`) makes it forecast past the end of what it
has seen, which is the only task anyone actually faces.

In [ ]:
rng = np.random.default_rng(8)
t = np.arange(400).reshape(-1, 1)                         # the time index, used as the feature
walk = np.cumsum(rng.normal(size=400))                    # the value drifts as time passes

random_time = cross_val_score(KNeighborsRegressor(5), t, walk,
                              cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()
forward = cross_val_score(KNeighborsRegressor(5), t, walk,
                          cv=TimeSeriesSplit(5), scoring="r2").mean()
print(f"  random split: R2 {random_time:+.3f}     forward (time-ordered) split: R2 {forward:+.3f}")

## Compare

1. Which one did the room split on? (It is usually 2 or 6.)
2. Of the ones that leak, which produced the **largest** gap when measured?
   Was it the one you expected?
3. Number 6 is the subtlest, and it is the one you will meet again on October 1
   and in every capstone. Why is the best of 200 CV scores not an unbiased
   estimate of anything?
4. Numbers 7 and 8 are the ones that will bite your capstone. Which of the
   datasets on the curated list have this structure?

## Exit ticket

> **Pick one leak from today. Describe the moment in a real deployment when you
> would discover that the number you reported was wrong.**

---

**Lab 2 is due Tuesday Sep 22, 11:59 pm.**

---

## Also in this neighborhood

📗 **Pipelines that resample.** If you do need to rebalance classes,
`imblearn.pipeline.Pipeline` applies the resampling step **inside** each training
fold, which is the only correct way to do it. Note that the honest AUC in §3 was
0.836 — *lower* than the inflated one, and lower than doing nothing clever at
all. Before reaching for resampling, try moving the decision threshold instead
(Week 5). It is usually the better answer and it cannot leak.

📗 **Nested cross-validation.** Workflow 6 in today's triage — reporting the best
of 200 CV scores — has a proper fix: an outer CV loop around the entire tuning
procedure. `cross_val_score(GridSearchCV(...), X, y, cv=outer)`. We do this
properly on October 1. Read ahead if today's #6 bothered you, and it should.

🚫 **SMOTE, and the imbalanced-learn recipe generally.** You will be taught this
and you will see it in tutorials: "the classes are imbalanced, so oversample
before modeling." Section 3 is what that costs when applied before the split,
and SMOTE is not exempt — its synthetic points are interpolations between real
ones, so they are not independent of the rows they were built from.

Class imbalance is also less often the problem than it is claimed to be. What
people usually mean is "accuracy is a bad metric here" and "the default 0.5
threshold is wrong for my decision." Both are true, and neither is fixed by
duplicating rows.

🎓 **Data provenance and versioning.** Today's leaks were all inside one file. In
a real project the dangerous ones come from *joins* — a feature table built later
than the outcome, a lookup silently refreshed. Tooling for tracking which version
of which table produced a model is a serious topic and a real job.